# Random Forest (Turkey, Pipeline B)

**Model**: `sklearn.ensemble.RandomForestRegressor` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: NO / **HP tuning**: YES — GridSearchCV(TimeSeriesSplit(5)) on train+val / **n_lags**: 4  
**10-model averaging** with seeds SEED+0 … SEED+9 / **n_jobs=1** (Windows constraint)


In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42; np.random.seed(SEED)
TRAIN_START = '1995-01-01'; TRAIN_END = '2011-12-31'; VAL_END = '2017-12-31'
TEST_START  = '2018-01-01'; TEST_END  = '2025-12-31'
N_LAGS = 4; N_MODELS = 10
VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('RF (Turkey) — Cat3, n_lags={}, scaling=NO, 10-model avg'.format(N_LAGS))


RF (Turkey) — Cat3, n_lags=4, scaling=NO, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {}'.format(len(features)))

# Phase A: HP tuning on train+val (1995-2017)
tune_data   = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat   = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat   = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)

feature_cols = [
    c for c in tune_flat.columns
    if c != 'date' and c != TARGET
    and any(c == f or c.startswith(f + '_') for f in features)
]

tscv      = TimeSeriesSplit(n_splits=5)
rf_base   = RandomForestRegressor(random_state=SEED, n_jobs=1)
rf_grid   = {'n_estimators': [200, 500], 'max_depth': [None, 8, 16], 'min_samples_leaf': [1, 3, 5]}
rf_search = GridSearchCV(rf_base, rf_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
rf_search.fit(tune_flat[feature_cols].values, tune_flat[TARGET].values)
best_params = rf_search.best_params_
print('Best RF params: {}'.format(best_params))


Features: 25


Best RF params: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 200}


In [3]:
# Phase B: Rolling test loop with 10-model averaging
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])
]['date'].tolist()

predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        vintage_date = pd_q + pd.DateOffset(months=offset)
        train_fl, tr_fl, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, vintage_date, N_LAGS
        )

        if len(train_fl) < 10 or len(tr_fl) != 1:
            predictions[vn].append(np.nan); continue

        try:
            models = []
            for k in range(N_MODELS):
                m = RandomForestRegressor(
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    min_samples_leaf=best_params['min_samples_leaf'],
                    random_state=SEED+k, n_jobs=1)
                m.fit(train_fl[feature_cols].values, train_fl[TARGET].values)
                models.append(m)

            preds  = [m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1

    if (i+1) % 8 == 0:
        print('  {} / {}'.format(i+1, len(test_quarters)))

print('Done. {} failures.'.format(failed))


  8 / 32


  16 / 32


  24 / 32


  32 / 32
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]})
    path = os.path.join(PREDICTIONS_DIR, 'rf_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))


Saved rf_m1.csv (32 rows) pred=[0.0072,0.0211]
Saved rf_m2.csv (32 rows) pred=[0.0071,0.0279]
Saved rf_m3.csv (32 rows) pred=[-0.0062,0.0252]
Saved rf_post1.csv (32 rows) pred=[-0.0103,0.0237]
Saved rf_post2.csv (32 rows) pred=[-0.0160,0.0229]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print('RF (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


RF (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.017544  MAE=0.012910  N=8
  m2  RMSFE=0.017439  MAE=0.012936  N=8
  m3  RMSFE=0.016865  MAE=0.015449  N=8
  post1  RMSFE=0.015994  MAE=0.014825  N=8
  post2  RMSFE=0.015496  MAE=0.014526  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.068442  MAE=0.040173  N=7
  m2  RMSFE=0.068031  MAE=0.039468  N=7
  m3  RMSFE=0.065099  MAE=0.040964  N=7
  post1  RMSFE=0.065162  MAE=0.041575  N=7
  post2  RMSFE=0.065598  MAE=0.042614  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.014374  MAE=0.011814  N=16
  m2  RMSFE=0.013557  MAE=0.010861  N=16
  m3  RMSFE=0.012146  MAE=0.008640  N=16
  post1  RMSFE=0.011994  MAE=0.008739  N=16
  post2  RMSFE=0.013510  MAE=0.010602  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.034852  MAE=0.018473  N=32
  m2  RMSFE=0.034511  MAE=0.017875  N=32
  m3  RMSFE=0.032873  MAE=0.017665  N=32
  post1  RMSFE=0.032784  MAE=0.017732  N=32
  post2  RMSFE=0.033139  MAE=0.018674  N=3